In [71]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import GradScaler, autocast
from tqdm import tqdm
import pandas as pd
import os
from PIL import Image

# Проверка устройства
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Пути к данным
train_csv_path = "/kaggle/input/dl-5-image-classification/train-labels.csv"
train_dir = "/kaggle/input/dl-5-image-classification/train"
test_dir = "/kaggle/input/dl-5-image-classification/test/test"

In [72]:
# Очистка train-labels.csv от отсутствующих файлов
df = pd.read_csv(train_csv_path)
df = df[df['image'].apply(lambda x: os.path.exists(os.path.join(train_dir, x)))]
df.to_csv("cleaned_train_labels.csv", index=False)  # сохраняем очищенный CSV
train_csv = "cleaned_train_labels.csv"

In [73]:
# Аугментации
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


In [74]:
# Загружаем CSV и фильтруем только существующие файлы
df = pd.read_csv(train_csv_path)
existing_files = set(os.listdir(train_dir))  # создаем множество файлов
df = df[df['image'].isin(existing_files)]  # оставляем только существующие
df.to_csv("cleaned_train_labels.csv", index=False)  # сохраняем очищенный CSV

train_csv = "cleaned_train_labels.csv"  # теперь используем его

In [75]:
class CustomDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None, train=True):
        self.img_dir = img_dir
        self.transform = transform
        self.train = train

        if train:
            self.data = pd.read_csv(csv_file)
            self.labels = sorted(self.data.label.unique())
            self.label_to_idx = {label: idx for idx, label in enumerate(self.labels)}
        else:
            # Вместо простого os.listdir()
            all_items = os.listdir(img_dir)
            images = [f for f in all_items if os.path.isfile(os.path.join(img_dir, f))]
            self.data = pd.DataFrame(images, columns=['image'])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_name = self.data.iloc[idx, 0]
        img_path = os.path.join(self.img_dir, img_name)

        # Если вдруг файла нет (на всякий случай)
        if not os.path.exists(img_path):
            return self.__getitem__((idx + 1) % len(self.data))

        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        if self.train:
            label = self.data.iloc[idx, 1]
            label_idx = self.label_to_idx[label]
            return image, torch.tensor(label_idx, dtype=torch.long)
        else:
            # Для теста
            return image, img_name

In [76]:
# Датасеты и загрузчики
datasets = {
    'train': CustomDataset(train_csv, train_dir, train_transforms, train=True),
    'test': CustomDataset(None, test_dir, test_transforms, train=False)
}

dataloaders = {
    'train': DataLoader(datasets['train'], batch_size=64, shuffle=True, num_workers=2),
    'test': DataLoader(datasets['test'], batch_size=64, shuffle=False, num_workers=2)
}

In [77]:
# Загрузка модели
def get_model():
    model = models.efficientnet_b0(weights=None)
    model.load_state_dict(torch.load("/kaggle/input/efficient/efficientnet_b0_rwightman-7f5810bc.pth"))
    for param in model.features.parameters():
        param.requires_grad = False
    
    num_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Linear(num_features, 256),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(256, 20)  # 20 классов
    )
    
    return model.to(device)

model = get_model()


Exception in thread QueueFeederThread:
Traceback (most recent call last):
  File "/usr/lib/python3.10/multiprocessing/queues.py", line 239, in _feed
    reader_close()
  File "/usr/lib/python3.10/multiprocessing/connection.py", line 177, in close
    self._close()
  File "/usr/lib/python3.10/multiprocessing/connection.py", line 361, in _close
    _close(self._handle)
OSError: [Errno 9] Bad file descriptor

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib/python3.10/threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.10/threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/lib/python3.10/multiprocessing/queues.py", line 271, in _feed
    queue_sem.release()
ValueError: semaphore or lock released too many times
<ipython-input-77-731a918bf29d>:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which 

In [78]:
# Оптимизатор и лр
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=3e-4, steps_per_epoch=len(dataloaders['train']), epochs=10)
criterion = nn.CrossEntropyLoss()
scaler = GradScaler()

<ipython-input-78-48e6e3350696>:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/usr/local/lib/python3.10/dist-packages/torch/amp/grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


In [81]:
# Функция обучения
def train_model(model, epochs=3):
    for epoch in range(epochs):
        model.train()
        total_loss, total_correct = 0, 0
        for images, labels in tqdm(dataloaders['train'], desc=f'Epoch {epoch+1}/{epochs}'):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast(device_type='cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total_loss += loss.item()
            total_correct += (outputs.argmax(1) == labels).sum().item()
        print(f'Train Loss: {total_loss/len(dataloaders["train"]):.4f}, Acc: {total_correct/len(datasets["train"]):.4f}')

train_model(model)
torch.save(model.state_dict(), "model.pth")
print("Model saved as model.pth")

Epoch 1/3: 100%|██████████| 259/259 [18:53<00:00,  4.38s/it]


Train Loss: 2.6016, Acc: 0.3169


Epoch 2/3: 100%|██████████| 259/259 [18:59<00:00,  4.40s/it]


Train Loss: 1.1558, Acc: 0.6986


Epoch 3/3: 100%|██████████| 259/259 [18:59<00:00,  4.40s/it]

Train Loss: 0.5897, Acc: 0.8194
Model saved as model.pth


In [83]:
# Предсказания на тесте
def predict(model, dataloader):
    model.eval()
    predictions = []
    with torch.no_grad():
        for images, img_names in tqdm(dataloader, desc='Predicting'):
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(1).cpu().numpy()
            predictions.extend(zip(img_names, preds))
    return predictions

In [84]:
# Запись предсказаний
predictions = predict(model, dataloaders['test'])
labels = datasets['train'].labels

submission = pd.DataFrame(predictions, columns=['image', 'label'])
submission['label'] = submission['label'].apply(lambda x: labels[x])
submission.to_csv("submission.csv", index=False)

print("Submission file saved!")

Predicting: 100%|██████████| 259/259 [17:30<00:00,  4.05s/it]

Submission file saved!


в качестве модели для классификации изображений была использована efficientnet-b0, загруженная с предобученными весами. предобученная основа была заморожена, а выходной слой заменен на полносвязную сеть, адаптированную под 20 классов.
для улучшения качества модели были применены следующие аугментации:

для обучающего набора:
	RandomResizedCrop(224, scale=(0.8, 1.0)) – случайное изменение размера изображения с кропом
	RandomHorizontalFlip() – случайное горизонтальное отражение
	ColorJitter(brightness=0.2, contrast=0.2) – изменение яркости и контрастности
	ToTensor() – преобразование изображения в тензор
	Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) – нормализация

для тестового набора:
	Resize((224, 224)) – изменение размера изображения
	ToTensor() – преобразование в тензор
	Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) – нормализация

 в процессе обучения использовалась CrossEntropyLoss и оптимизатор AdamW с шагом 3e-4, а также планировщик OneCycleLR. запустила обучение на 3 эпохи, после чего модель достигла accuracy ~ 81.9% на обучающей выборке.

после трех эпох обучения модель достигла accuracy: 0.8194 на тренировочном наборе. затем были сделаны предсказания на тестовом наборе, и итоговый public score на kaggle составил 0.87079, что является хорошим результатом для данной задачи.